In [ ]:
"""
PTQ Best Config — static_minmax — ResNet-50 / CIFAR-10
=======================================================
Runs ONLY the best-performing PTQ configuration identified from the sweep:
static_minmax (FX-graph, MinMax observer, INT8 weights + activations)

Why static_minmax beats dynamic:
Dynamic PTQ quantizes ONLY nn.Linear weights → qint8.
Conv2d layers (which dominate ResNet-50 compute) remain FP32.
Result: negligible size reduction, negligible speedup, same accuracy.

Static PTQ quantizes Conv2d + Linear + residual adds (weights + activations).
Result: 4× smaller, 3× faster batch throughput, accuracy drop < 0.0001.

Param counting fix vs v2:
FIX-2 in v2 used data_ptr() for dedup, which caused:
    - FP32/dynamic models: params DOUBLED (different tensors, same storage offset)
    - Static FX models: params too LOW (over-deduplication)

Correct approach used here:
    - For parameters(): use id(p) — each Parameter object is unique, id() is safe.
    - For FX .weight() callables: use a (module_path, 'weight') string key —
    avoids data_ptr() aliasing while still preventing the original double-count
    that occurred when the same module was visited twice via named_modules().

Mathematical foundation (MinMax observer):
S = (x_max − x_min) / (2^8 − 1)   [255 quantization levels for INT8]
Z = round(−x_min / S)              [zero-point: maps real 0 to integer grid]
Q(x) = clamp(round(x / S) + Z, 0, 255)
x̂   = S · (Q(x) − Z)              [reconstruction at inference]
|ε|  ≤ S/2                         [max error bounded by half step size]

MinMax observer: x_min/x_max taken from the global min/max across all
calibration batches. Simple but sensitive to outliers. Works well for
ResNet activations which have relatively tight distributions.

Output: __1__PTQ_best.json
        __1__PTQ_best_model/resnet50_static_minmax.pt
"""

# %%
import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.quantization as tq
from torch.ao.quantization import QConfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH = "../__2__baseline_resnet50_cifar10.pth"
OUTPUT_JSON         = "__1__PTQ_CPU.json"
MODEL_SAVE_DIR      = "__1__saved_models_PTQ_CPU"

DEVICE         = torch.device("cpu")
BATCH_SIZE     = 128
NUM_WORKERS    = 2
NUM_CLASSES    = 10
CALIB_SIZE     = 1024
CALIB_BATCHES  = 8
INFERENCE_RUNS = 200
WARMUP_RUNS    = 20

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Best QConfig: MinMax observer ─────────────────────────────────────────────
# Activation: per-tensor affine, MinMax range → quint8 (unsigned, 0-255)
# Weight:     per-channel symmetric, MinMax range → qint8 (signed, -128..127)
#             Per-channel gives each filter its own scale, reducing error vs
#             per-tensor which uses one global scale for all output channels.
BEST_QCONFIG = QConfig(
    activation=tq.MinMaxObserver.with_args(
        dtype=torch.quint8, qscheme=torch.per_tensor_affine),
    weight=tq.PerChannelMinMaxObserver.with_args(
        dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
)

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model().cpu()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                    download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=False)

def get_calib_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = torchvision.datasets.CIFAR10(root="../data", train=True,
                                        download=True, transform=transform)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=False)

In [ ]:
# ── Utilities ─────────────────────────────────────────────────────────────────
def disk_size_mb(model, use_state_dict=False):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict() if use_state_dict else model, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def count_params(model, model_label=""):
    """
    Correct parameter counting for both FP32 and FX-quantized models.

    Strategy:
    - For standard nn.Parameter tensors: deduplicate by id(p).
        id() is safe here because each Parameter is a distinct Python object,
        even if two parameters share underlying storage (extremely rare in
        standard ResNet architectures).

    - For FX-quantized layers with a .weight() callable: deduplicate by
        a string key "(module_name, 'weight')" constructed from named_modules().
        This prevents counting the same quantized weight twice when a module
        appears in the graph under multiple traversal paths, WITHOUT the
        data_ptr() aliasing bug that caused doubling in v2.

    Why NOT data_ptr():
    Different Parameter objects can have the same data_ptr() if they are
    views/slices of the same storage (e.g. grouped convolutions, some fused
    ops). Using data_ptr() as a dedup key merges genuinely separate parameters,
    causing undercounting. id() on the Python object is the correct key for
    nn.Parameter. For FX callable weights (which return new tensors on each
    call), a module-path string key is used instead.
    """
    total, nonzero   = 0, 0
    seen_param_ids   = set()   # for nn.Parameter objects
    seen_module_keys = set()   # for FX .weight() callable modules

    for mod_name, module in model.named_modules():
        # ── Standard parameters (FP32, dynamic PTQ, bias terms) ──────────────
        for param_name, p in module.named_parameters(recurse=False):
            pid = id(p)
            if pid in seen_param_ids:
                continue
            seen_param_ids.add(pid)
            n        = p.numel()
            total   += n
            nonzero += int((p != 0).sum().item())

        # ── FX quantized weight callable ──────────────────────────────────────
        # After convert_fx(), quantized ops expose weights via .weight() which
        # returns a freshly dequantized FloatTensor on every call.
        # We cannot use id() or data_ptr() on these — use the module path instead.
        if hasattr(module, 'weight') and callable(module.weight):
            key = (mod_name, 'weight')
            if key in seen_module_keys:
                continue
            seen_module_keys.add(key)
            try:
                w = module.weight()   # dequantized → FloatTensor
                if isinstance(w, torch.Tensor) and w.numel() > 0:
                    total   += w.numel()
                    nonzero += int((w != 0).sum().item())
            except Exception:
                pass

    if model_label:
        print(f"      count_params [{model_label}]: "
            f"total={total:,}  nonzero={nonzero:,}  "
            f"sparsity={1 - nonzero/max(total,1):.4f}")
    return {"params_total": int(total), "params_nonzero": int(nonzero)}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        preds.extend(model(inputs).argmax(1).numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_cpu_throughput(model, n=INFERENCE_RUNS):
    model = model.cpu().eval()
    dummy_batch  = torch.randn(BATCH_SIZE, 3, 32, 32)
    dummy_single = torch.randn(1,          3, 32, 32)

    with torch.no_grad():
        for _ in range(WARMUP_RUNS):
            model(dummy_batch)
            model(dummy_single)

    batch_timings, single_timings = [], []
    with torch.no_grad():
        for _ in range(n):
            t0 = time.perf_counter()
            model(dummy_batch)
            batch_timings.append((time.perf_counter() - t0) * 1000)

        for _ in range(n):
            t0 = time.perf_counter()
            model(dummy_single)
            single_timings.append((time.perf_counter() - t0) * 1000)

    batch_ms  = float(np.mean(batch_timings))
    single_ms = float(np.mean(single_timings))
    return {
        "single_img_cpu_ms"  : round(single_ms, 4),
        "batch128_cpu_ms"    : round(batch_ms, 4),
        "per_img_cpu_ms"     : round(batch_ms / BATCH_SIZE, 4),
        "throughput_imgs_sec": round(BATCH_SIZE / (batch_ms / 1000), 1),
        "timing_method"      : "wall-clock (time.perf_counter) — CPU",
    }

def compute_flops(model):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = profile(model, inputs=(dummy,), verbose=False)
        return float(macs * 2 / 1e9)
    except Exception:
        return -1.0

@torch.no_grad()
def calibrate(model, loader, max_batches=CALIB_BATCHES):
    model.eval()
    for i, (inputs, _) in enumerate(loader):
        model(inputs)
        if i + 1 >= max_batches:
            break


In [ ]:
# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    SEPARATOR = "=" * 65

    print(f"\n{SEPARATOR}")
    print("  PTQ Best Config — static_minmax — ResNet-50 / CIFAR-10")
    print("  Observer: MinMax  |  INT8 weights + activations (Conv + Linear)")
    print(f"  Calibration: {CALIB_SIZE} images  |  Runs: {INFERENCE_RUNS}  |  Warmup: {WARMUP_RUNS}")
    print(SEPARATOR)

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    # ── FP32 Baseline ─────────────────────────────────────────────────────────
    print("\n  [1/2] FP32 Baseline")
    baseline_metrics = evaluate(fp32_model, test_loader)
    base_infer       = measure_cpu_throughput(fp32_model)
    base_disk        = disk_size_mb(fp32_model, use_state_dict=False)
    base_flops       = compute_flops(fp32_model)
    base_params      = count_params(fp32_model, model_label="fp32_baseline")

    print(f"        Acc={baseline_metrics['accuracy']:.4f}  "
        f"Disk={base_disk:.2f} MB  "
        f"Batch={base_infer['batch128_cpu_ms']:.1f} ms  "
        f"GFLOPs={base_flops:.3f}")

    # ── static_minmax PTQ ─────────────────────────────────────────────────────
    print("\n  [2/2] Static PTQ — MinMax observer")
    print("        Fusing Conv-BN-ReLU → computing S, Z from calibration data ...")

    example = torch.randn(1, 3, 32, 32)
    model   = copy.deepcopy(fp32_model).cpu().eval()

    prepared = prepare_fx(model, {"": BEST_QCONFIG}, example)
    calibrate(prepared, calib_loader)
    q_model = convert_fx(prepared)
    q_model.eval()

    print("        Evaluating on test set ...")
    q_metrics  = evaluate(q_model, test_loader)
    q_infer    = measure_cpu_throughput(q_model)
    q_disk     = disk_size_mb(q_model, use_state_dict=False)
    q_flops    = compute_flops(q_model)
    # Use base_flops if thop returns ~0 for FX model (no hookable Conv2d/Linear)
    if q_flops < 0.01:
        q_flops = base_flops
    q_params   = count_params(q_model, model_label="static_minmax")

    # ── Build output ──────────────────────────────────────────────────────────
    acc_drop    = baseline_metrics["accuracy"] - q_metrics["accuracy"]
    batch_speedup = base_infer["batch128_cpu_ms"] / q_infer["batch128_cpu_ms"]
    size_ratio  = base_disk / q_disk

    output = {
        "fp32_baseline": {
            "accuracy"      : round(baseline_metrics["accuracy"],  6),
            "precision"     : round(baseline_metrics["precision"], 6),
            "recall"        : round(baseline_metrics["recall"],    6),
            "f1"            : round(baseline_metrics["f1"],        6),
            "params"        : int(base_params["params_total"]),
            "params_nonzero": int(base_params["params_nonzero"]),
            "size_mb"       : round(base_disk, 6),
            "flops_G"       : round(base_flops, 9),
            "inference_ms"  : base_infer,
        },
        "static_minmax": {
            "accuracy"      : round(q_metrics["accuracy"],  6),
            "precision"     : round(q_metrics["precision"], 6),
            "recall"        : round(q_metrics["recall"],    6),
            "f1"            : round(q_metrics["f1"],        6),
            "params"        : int(q_params["params_total"]),
            "params_nonzero": int(q_params["params_nonzero"]),
            "size_mb"       : round(q_disk, 6),
            "flops_G"       : round(q_flops, 9),
            "inference_ms"  : q_infer,
            "notes": (
                "single_img_cpu_ms higher than FP32 is expected — FX op dispatch "
                "overhead dominates at batch_size=1. Batch throughput is the "
                "meaningful deployment metric."
            ),
        },
        "_meta": {
            "best_config"     : "static_minmax",
            "observer"        : "MinMaxObserver (per-tensor activation, per-channel weight)",
            "acc_drop"        : round(acc_drop, 6),
            "batch_speedup_x" : round(batch_speedup, 4),
            "size_reduction_x": round(size_ratio, 4),
            "calib_images"    : CALIB_SIZE,
            "calib_batches"   : CALIB_BATCHES,
            "inference_runs"  : INFERENCE_RUNS,
            "warmup_runs"     : WARMUP_RUNS,
            "flops_note": (
                "PTQ does not change the computation graph — GFLOPs identical to "
                "FP32 baseline. INT8 batch speedup comes from VNNI/SIMD packing "
                "(4 INT8 MACs per instruction vs 1 FP32 FMA), not fewer operations."
            ),
        },
    }

    # ── Save JSON ─────────────────────────────────────────────────────────────
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    # ── Save model ────────────────────────────────────────────────────────────
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
    model_path = os.path.join(MODEL_SAVE_DIR, "resnet50_static_minmax.pt")
    torch.save(q_model, model_path)
    saved_mb = os.path.getsize(model_path) / 1024 ** 2
    print(f"        Saved → {model_path}  ({saved_mb:.2f} MB)")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{SEPARATOR}")
    print("  RESULTS SUMMARY")
    print(SEPARATOR)
    print(f"  {'Metric':<28} {'FP32':>12}  {'static_minmax':>14}")
    print(f"  {'-'*60}")
    print(f"  {'Accuracy':<28} {baseline_metrics['accuracy']:>12.4f}  {q_metrics['accuracy']:>14.4f}")
    print(f"  {'F1 (macro)':<28} {baseline_metrics['f1']:>12.4f}  {q_metrics['f1']:>14.4f}")
    print(f"  {'Params (total)':<28} {base_params['params_total']:>12,}  {q_params['params_total']:>14,}")
    print(f"  {'Params (nonzero)':<28} {base_params['params_nonzero']:>12,}  {q_params['params_nonzero']:>14,}")
    print(f"  {'Disk size (MB)':<28} {base_disk:>12.2f}  {q_disk:>14.2f}")
    print(f"  {'Batch latency (ms)':<28} {base_infer['batch128_cpu_ms']:>12.1f}  {q_infer['batch128_cpu_ms']:>14.1f}")
    print(f"  {'Throughput (imgs/s)':<28} {base_infer['throughput_imgs_sec']:>12.1f}  {q_infer['throughput_imgs_sec']:>14.1f}")
    print(f"  {'Single-img latency (ms)':<28} {base_infer['single_img_cpu_ms']:>12.1f}  {q_infer['single_img_cpu_ms']:>14.1f}")
    print(f"  {'GFLOPs':<28} {base_flops:>12.4f}  {q_flops:>14.4f}")
    print(f"  {'-'*60}")
    print(f"  Accuracy drop   : {acc_drop:+.4f}")
    print(f"  Batch speedup   : {batch_speedup:.2f}×")
    print(f"  Size reduction  : {size_ratio:.2f}×")
    print(f"\n  JSON  → {OUTPUT_JSON}")
    print(f"  Model → {model_path}")
    print(SEPARATOR + "\n")

    return output, q_model





  PTQ Best Config — static_minmax — ResNet-50 / CIFAR-10
  Observer: MinMax  |  INT8 weights + activations (Conv + Linear)
  Calibration: 1024 images  |  Runs: 200  |  Warmup: 20

  [1/2] FP32 Baseline
      count_params [fp32_baseline]: total=23,520,842  nonzero=23,520,842  sparsity=0.0000
        Acc=0.9320  Disk=90.05 MB  Batch=7997.7 ms  GFLOPs=2.623

  [2/2] Static PTQ — MinMax observer
        Fusing Conv-BN-ReLU → computing S, Z from calibration data ...


W0505 13:51:45.376000 12772 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


        Evaluating on test set ...
      count_params [static_minmax]: total=23,467,712  nonzero=20,599,612  sparsity=0.1222
        Saved → __1__saved_models_PTQ_CPU\resnet50_static_minmax.pt  (23.01 MB)

  RESULTS SUMMARY
  Metric                               FP32   static_minmax
  ------------------------------------------------------------
  Accuracy                           0.9320          0.9329
  F1 (macro)                         0.9319          0.9328
  Params (total)                 23,520,842      23,467,712
  Params (nonzero)               23,520,842      20,599,612
  Disk size (MB)                      90.05           22.99
  Batch latency (ms)                 7997.7           442.5
  Throughput (imgs/s)                  16.0           289.3
  Single-img latency (ms)              45.8           199.6
  GFLOPs                             2.6232          2.6232
  ------------------------------------------------------------
  Accuracy drop   : -0.0009
  Batch speedup   : 18

In [ ]:
if __name__ == "__main__":
    report, best_model = main()